# 00. Setup and data verification

- D1: AI-Hub dataset 138 (registered access)
- D2: Hong Kong field recordings, DOI: [10.17632/hkn8mxcjyz.1](https://doi.org/10.17632/hkn8mxcjyz.1)
- D3: Aghashahi testbed, DOI: [10.17632/tbrnp6vrnj.1](https://doi.org/10.17632/tbrnp6vrnj.1)

In [1]:
import sys, os, platform, json
sys.path.insert(0, os.path.abspath(".."))  
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation as ev, reporting as rp
cfg.apply_mpl_style(); cfg.banner()

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000


## 1. Environment

In [2]:
env = {"python": platform.python_version(), "platform": platform.platform()}
for mod in ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "librosa",
            "soundfile", "torch", "torchaudio", "transformers"]:
    try:
        m = __import__(mod)
        env[mod] = getattr(m, "__version__", "?")
    except ImportError:
        env[mod] = "not installed"
try:
    import torch
    env["cuda"] = bool(torch.cuda.is_available())
except ImportError:
    pass
for k, v in env.items():
    print(f"{k:12s} {v}")
la.save_json(env, "environment.json")

python       3.12.14
platform     Windows-11-10.0.26200-SP0
numpy        2.5.2
pandas       2.3.3
scipy        1.18.1
sklearn      1.9.0
matplotlib   3.11.1
librosa      1.0.0
soundfile    0.14.0
torch        2.13.0+cpu
torchaudio   2.11.0+cpu
transformers 5.16.1
cuda         False


WindowsPath('C:/Users/ecredible/Desktop/LeakAudit_revision1/LeakAudit/results/environment.json')

## 2. Data paths

In [ ]:
# cfg.set_root(r"C:\path\to\LeakAudit")
print("ROOT   =", cfg.ROOT)
print("D1     =", cfg.D1)
print("D2     =", cfg.D2)
print("D3     =", cfg.D3)

ROOT   = C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit
D1     = C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit\data_raw\D1_aihub
D2     = C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit\data_raw\D2_hongkong
D3     = C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit\data_raw\D3_testbed


## 3. File inventory

Expected: D1 10 CSV / D2 140 WAV + 20 XLSX / D3 160 CSV + 122 RAW (+ 1 py file)

In [4]:
io_utils.diagnose(cfg.D1, "D1", {".csv"})
io_utils.diagnose(cfg.D2, "D2", {".wav", ".xlsx"})
io_utils.diagnose(cfg.D3, "D3", {".csv", ".raw"})

[D1] path   = C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit\data_raw\D1_aihub
[D1] exists = True
[D1] files  = 10
.csv    10
[D1] target-extension files ['.csv'] = 10
[D1] top-level entries: ['1.옥외누수(out-training).csv', '1.옥외누수(out-validation).csv', '2.옥내누수(in-training).csv', '2.옥내누수(in-validation).csv', '3.기계.전기음(noise-training).csv', '3.기계.전기음(noise-validation).csv', '4.환경음(other-training).csv', '4.환경음(other-validation).csv', '5.정상음(normal-training).csv', '5.정상음(normal-validation).csv']
    1.옥외누수(out-training).csv
    1.옥외누수(out-validation).csv
    2.옥내누수(in-training).csv
    2.옥내누수(in-validation).csv
    3.기계.전기음(noise-training).csv
[D2] path   = C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit\data_raw\D2_hongkong
[D2] exists = True
[D2] files  = 160
.wav     140
.xlsx     20
[D2] target-extension files ['.wav', '.xlsx'] = 160
[D2] top-level entries: ['Hydrophones', 'MEMS Accelerometers', 'Noise Loggers']
    Hydrophones/Leak/1.1.01.0330.wav
    Hydrophones/Leak

## 4. Checksums

In [5]:
man = io_utils.write_data_manifest()
print(man.groupby("dataset").agg(files=("path", "size"), bytes=("bytes", "sum")))
d1_tab = man[man.dataset == "D1"].copy()
d1_tab["file"] = d1_tab.path.str.split("/").str[-1]
d1_tab[["file", "bytes", "sha256"]].to_csv(cfg.RESULTS / "D1_sha256_table.csv", index=False)
print()
print(d1_tab[["file", "bytes", "sha256"]].to_string(index=False))
conv = man[man.path.str.contains("convert", case=False)]
if len(conv):
    print("\nprovider conversion script:", conv.iloc[0].path, conv.iloc[0].sha256)
print(f"\nwritten: {cfg.ROOT / 'data_manifest.txt'}")

         files       bytes
dataset                   
D1          10    90046960
D2         160   257238788
D3         283  3881932294

                          file    bytes                                                           sha256
      1.옥외누수(out-training).csv 23057847 fa4f4b602f11e447207910701449f5c40b3a231ba2528756ea94d1aec05d0eb7
    1.옥외누수(out-validation).csv  2894531 2796edcaac92cb5ad449c8a1a77286e23636004bdbdadc137bd80707935ff765
       2.옥내누수(in-training).csv 17666329 b2283fcabf6fe56dce6c39ffda59c377903cf21c99dcc0e516130853802ef76f
     2.옥내누수(in-validation).csv  2198345 0e8f6cb6d9eac7317b74d5332191b39a928fe06102141cf2023226b8dcaa5387
  3.기계.전기음(noise-training).csv  6267068 9ced28733eab3389459a6c2192f3446e8b52afe72b304f96461a01d6ac95d81f
3.기계.전기음(noise-validation).csv   781679 c254c71bda78ae7017ee7b5ff72e854ed347c94b591847b128e334ba490a5576
     4.환경음(other-training).csv  9231547 0f02c76c891ded6a39b3df9c0d7fb8f9801061790e97d13ed6e0dfe1475d7b17
   4.환경음(other-validatio

## 5. Pipeline specification and protocol catalogue

In [6]:
la.save_json(cfg.PIPELINE, "pipeline_spec.json")
ev.PROTOCOL_CATALOGUE.to_csv(cfg.RESULTS / "protocol_catalogue.csv", index=False)
pd.set_option("display.max_colwidth", 90)
print(ev.PROTOCOL_CATALOGUE.to_string(index=False))

dataset              protocol                                                              held_out                                                                                                      deployment_setting        unit
     D1       P1_naive_random                                               nothing (rows shuffled)     none: rows of the same sensor and date-series sit on both sides; an upper bound that includes identity memorisation      record
     D1       P2_sensor_indep                                                          sensor (sid)                                                              a new sensor at an already-instrumented site (site shared)      record
     D1         P3_site_indep                                                                  site                                                                 a new site, any period (sites unseen, time not ordered)      record
     D1      P4_site_temporal                                 site + tim

## 6. Result log reset


In [7]:
rp.reset_ledger()
print("ledger reset:", cfg.RESULTS / rp.LEDGER)

ledger reset: C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit\results\revision_ledger.jsonl
